# ZETTY RBA — 현재 Colab VM 복구 최종본

**아래 코드 셀 하나만 실행하거나 전체를 새 Colab 코드 셀에 복붙하세요.** 이전 셀 변수에 의존하지 않습니다.

- 기본 경로: `/content/zetty-rba-20260927-082657-588360/prepared/rba`
- 기존 전처리 배열을 재사용하고 `colab-gpu-v2`로 새 학습을 시작합니다. 오류가 난 가중치 분위수 구성은 CPU, 트리 학습·예측은 CUDA입니다.
- Drive 용량 초과를 피하기 위해 **Drive에 쓰지 않습니다**. 완료되면 결과 ZIP 다운로드가 시작됩니다. 다운로드를 확인하기 전 VM을 종료하지 마세요.
- 기존 학습 프로세스가 종료된 현재 상황용입니다. 학습이 실행 중일 때 이 셀을 중복 실행하지 마세요.
- 기존 VM과 설치된 GPU 의존성을 사용합니다. 새 VM에서 처음 시작하는 일반 fresh/prepared/resume 절차는 `zetty_rba_colab_full.ipynb`에 보존했습니다.

전체 T4 데이터에 대한 수정본 성공 여부는 이번 실행으로 확인해야 합니다. Mac 로컬 학습은 실행하지 않습니다.


In [ ]:
# 이 셀 전체를 현재 Google Colab의 새 코드 셀에 복붙해서 실행하세요.
# 기존 학습 셀 재실행 / 런타임 삭제 없이 실행합니다.
import os
import platform
import runpy
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

assert platform.system() == 'Linux' and os.environ.get('COLAB_RELEASE_TAG'), (
    'Google-hosted Colab에서 실행하세요. Mac/IntelliJ 로컬 학습은 금지합니다.'
)
assert sys.version_info[:2] in ((3, 12), (3, 13)), f'지원하지 않는 Python: {sys.version}'
assert shutil.which('nvidia-smi'), (
    'GPU가 연결되지 않았습니다. 기존 VM 파일을 보존한 뒤 Colab GPU 연결을 확인하세요.'
)

# 현재 실행의 전처리 배열입니다. 기존 VM을 유지했다면 수정하지 마세요.
PREPARED = Path('/content/zetty-rba-20260927-082657-588360/prepared/rba')
required = ['dataset.json', 'x.bin', 'times.bin', 'entities.bin', 'labels.bin']
missing = [name for name in required if not (PREPARED/name).is_file()]
assert not missing, (
    f'전처리 파일이 없습니다: {PREPARED} / {missing}. '
    '새 VM이라면 Drive 원본 prepared/rba를 VM에 복원하고 PREPARED를 바꾸세요.'
)
subprocess.run([
    'nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'
], check=True)

# 의존성은 앞서 설치한 GPU lock 그대로 사용합니다. 설치 누락은 학습 전에 알립니다.
subprocess.run([
    sys.executable, '-c',
    'import numpy, scipy, sklearn, joblib, threadpoolctl, pyarrow, xgboost, cupy; print("GPU dependencies: import OK")'
], check=True)

stamp = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
REPO = Path('/content') / f'zetty-recovery-code-{stamp}'
subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', 'feature/public-data-training-star',
    'https://github.com/jjyj0203/uba-analyzer.git', str(REPO)
], check=True)
print('Code commit:', subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True
).strip(), flush=True)
print('Profile: colab-gpu-v2 / 기존 전처리 재사용 / Drive 백업 없음', flush=True)

# 새 출력·실시간 로그·CPU quantile → CUDA 학습·완료 ZIP 다운로드.
# 출력 경로 OUTPUT 및 LOG는 실행 시작 시 화면에 표시됩니다.
previous_argv = sys.argv[:]
try:
    sys.argv = [str(REPO/'scripts/ml/recover_colab.py'), '--prepared', str(PREPARED)]
    runpy.run_path(sys.argv[0], run_name='__main__')
finally:
    sys.argv = previous_argv


## 진행 확인

`phase=quantile, quantile_device=cpu` 다음 `phase=boosting, device=cuda:0` 및 `boosting_round`가 표시됩니다. 모델별 calibration/validation과 final_test가 끝나야 COMPLETED입니다.

오류가 나면 화면에 나온 `LOG` 파일과 기존 배열을 보존하세요. `RUNNING` 문구만으로 프로세스가 살아 있다고 판단하지 않습니다. 결과 ZIP은 모델·평가·실행 기록을 포함하며 전처리 배열은 포함하지 않습니다.

STAR 및 검증: `docs/ml/12-colab-recovery/README.md`, `RESULTS.md`.
